# imports

In [ ]:
install.packages("lubridate", repos = "https://cloud.r-project.org")

In [ ]:
library(dplyr)
library(MatchIt)
library(readr)
library(stringr)
library(tableone)
library(tibble)
library(tidyr)
library(ggplot2)
library(cobalt)
library(purrr)
library(cluster)
library(sandwich)
library(lmtest)
library(sensemakr)
library(EValue)
library(rbounds)
library(patchwork)
library(brglm2)
library(ggtext)
library(survival)
library(marginaleffects)
library(lubridate)


In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)

# ─────────────────────────────────────────────
# Color palette
# ─────────────────────────────────────────────
COL_BLUE <- "#2D6A9F"   # Steel blue (treatment)
COL_GOLD <- "#E09B3D"   # Warm gold (control)
COL_GRAY <- "#B0B0B0"   # Light gray (not significant)
COL_ROSE <- "#C45B5B"   # Muted rose (before matching)

# ─────────────────────────────────────────────
# Shared helpers
# ─────────────────────────────────────────────


# Variable name translations for love plots (column names + Hebrew factor levels)
love_var_translate <- function(nms) {
  translations <- c(
    "Criminal_record"     = "Criminal Record",
    "marital_status"      = "Marital Status",
    "Defendants_Ethnic"   = "Defendant Ethnicity",
    "judge_ethnic"        = "Judge Ethnicity",
    "total_amount"        = "Total Amount (log)",
    "production_offence"  = "Production Offense",
    "laboratory"          = "Laboratory",
    "self_use"            = "Self Use",
    "sold_to_agent"       = "Sold to Agent",
    "regret"              = "Regret",
    "confession"          = "Guilty Plea",
    "min_punishment"      = "Lower Bounds",
    "max_punishment"      = "Upper Bounds",
    "use_offence"         = "Possession Offense",
    "importation_offence" = "Trafficking Offense",
    "district"            = "District",
    "age"                 = "Age",
    "year"                = "Year",
    "Court"               = "Court Type",
    "responsabillity"     = "Responsibility",
    "לא ידוע"               = "Unknown",
    "ללא עבר פלילי" = "No Criminal Record",
    "עבר פלילי כבד" = "Severe Criminal Record",
    "עבר פלילי קל"   = "Minor Criminal Record",
    "גרוש"                 = "Divorced",
    "נשוי"                 = "Married",
    "רווק"                 = "Single",
    "מחוזי"           = "District Court",
    "שלום"                 = "Magistrate Court"
  )
  for (old in names(translations)) {
    nms <- gsub(old, translations[old], nms, fixed = TRUE)
  }
  nms
}

# Academic color palette (grayscale-friendly)
theme_academic <- function() {
  theme_gray() +
    theme(
      text = element_text(size = 11),
      plot.title = element_text(size = 12, face = "bold", hjust = 0.5),
      plot.background = element_rect(fill = "white", color = NA),
      axis.title = element_text(size = 11),
      axis.text = element_text(size = 10, color = "black"),
      legend.position = "top",
      legend.title = element_blank()
    )
}


# ─────────────────────────────────────────────
#  Propensity Score Distribution
# ─────────────────────────────────────────────

plot_ps_distribution_boxplot <- function(matched_df, method_label = "", 
                                        file_path = NULL, width = 8, height = 4) {
  
  if (!("distance" %in% names(matched_df)) || !("treated" %in% names(matched_df))) {
    message("No distance/treated columns found.")
    return(invisible(NULL))
  }
  
  df_plot <- matched_df %>%
    filter(!is.na(distance)) %>%
    mutate(group = ifelse(treated == 1, "Treatment", "Control"))
  
  p <- ggplot(df_plot, aes(x = group, y = distance, fill = group)) +
    geom_boxplot(outlier.size = 1) +
    scale_fill_manual(values = c("Control" = COL_GOLD, "Treatment" = COL_BLUE)) +
    labs(title = paste("Propensity Score Distribution —", method_label),
         x = NULL, y = "Propensity Score") +
    theme_academic() +
    theme(legend.position = "none")
  
  # Save if file path provided
  if (!is.null(file_path)) {
    ggsave(file_path, plot = p, width = width, height = height, dpi = 300, bg = "white")
  }
  
  return(p)
}









# Main Analaysis 

In [ ]:
# =============================================================================
# SHARED HELPERS
# =============================================================================

is_binary_01 <- function(x) {
  if (is.data.frame(x) || is.list(x)) return(FALSE)
  ux <- unique(x[!is.na(x)])
  length(ux) == 2 && all(as.numeric(sort(ux)) == c(0, 1))
}

gelman_scale_2sd <- function(x) {
  m <- mean(x, na.rm = TRUE)
  s <- sd(x,   na.rm = TRUE)
  if (is.na(s) || s == 0) return(x)
  (x - m) / (2 * s)
}

apply_gelman_scaling <- function(df, covars) {
  for (v in covars) {
    x <- df[[v]]
    df[[v]] <- if (is.factor(x) || is.character(x)) {
      as.factor(x)
    } else if (is.logical(x)) {
      as.integer(x)
    } else if (is.numeric(x) || is.integer(x)) {
      if (is_binary_01(x)) as.integer(x) else gelman_scale_2sd(as.numeric(x))
    } else {
      as.factor(x)
    }
  }
  df
}

compute_cohens_d <- function(x_t, x_c) {
  n_t <- sum(!is.na(x_t))
  n_c <- sum(!is.na(x_c))
  if (n_t < 2 || n_c < 2) return(NA_real_)

  sd_pooled <- sqrt(
    ((n_t - 1) * var(x_t, na.rm = TRUE) +
     (n_c - 1) * var(x_c, na.rm = TRUE)) / (n_t + n_c - 2)
  )
  if (!is.finite(sd_pooled) || sd_pooled == 0) return(NA_real_)

  (mean(x_t, na.rm = TRUE) - mean(x_c, na.rm = TRUE)) / sd_pooled
}

# Fit lm or glm depending on outcome type
fit_model <- function(form, data, is_binary) {
  if (is_binary) glm(form, data = data, family = binomial())
  else           lm( form, data = data)
}

# Add pair_id column from subclass (if exists) or row number
add_pair_id <- function(df) {
  if ("subclass" %in% names(df)) rename(df, pair_id = subclass)
  else mutate(df, pair_id = as.character(row_number()))
}

# Significance stars from p-value
sig_label <- function(p) {
  case_when(
    p <= 0.001 ~ "***",
    p <= 0.01  ~ "**",
    p <= 0.05  ~ "*",
    TRUE       ~ ""
  )
}

# OR and its CI (NA for continuous outcomes)
extract_or <- function(is_binary, robust_coef, ci_robust) {
  if (!is_binary) return(list(val = NA_real_, lo = NA_real_, hi = NA_real_))
  list(
    val = exp(robust_coef["treated", 1]),
    lo  = exp(ci_robust["treated",   1]),
    hi  = exp(ci_robust["treated",   2])
  )
}


# =============================================================================
# MAIN FUNCTION
# =============================================================================

run_matching <- function(
    df,
    group_col,
    targets_cols,
    output_dir,
    treatment,
    control,
    caliper_q         = 0.9,
    caliper_ps_sd     = 0.2,
    match_mode        = c("gower", "logit", "exact_cem", "no_caliper", "custom"),
    all_covars        = NULL,
    fact_covars       = NULL,
    demo_covars       = NULL,
    gower_demo_weight = 1,
    gower_fact_weight = 4,
    only_fact_vars    = FALSE,
    binary_targets    = NULL,
    interaction_var   = "ethnic",
    bonferroni_k      = 1,
    war_interaction   = FALSE
) {

  match_mode <- match.arg(match_mode)

  # ---------------------------------------------------------------------------
  # 1. COVARIATE SETUP
  # ---------------------------------------------------------------------------

  base_covars <- if (only_fact_vars) fact_covars else all_covars
  base_covars <- setdiff(base_covars, group_col)

  # ---------------------------------------------------------------------------
  # 2. DATA PREPARATION
  # ---------------------------------------------------------------------------

  df_before <- df %>%
    filter(.data[[group_col]] %in% c(treatment, control)) %>%
    drop_na(any_of(c(group_col, base_covars, targets_cols))) %>%
    mutate(treated = as.integer(.data[[group_col]] == treatment))

  df_group <- df_before %>%
    mutate(total_amount = log(total_amount + 0.001))

  # Factorize categoricals (keep year numeric)
  for (v in base_covars) {
    if (!v %in% names(df_group) || v == "year") next
    if (is.character(df_group[[v]]) || length(unique(df_group[[v]])) < 10) {
      df_group[[v]] <- as.factor(df_group[[v]])
      df_before[[v]] <- as.factor(df_before[[v]])  # apply to both
    }
  }

  covars <- intersect(base_covars, names(df_group))

  # Drop unused factor levels
  for (v in covars)
    if (is.factor(df_group[[v]])) df_group[[v]] <- droplevels(df_group[[v]])

  # Remove constant covariates
  is_constant <- function(x)
    if (is.factor(x)) nlevels(x) < 2 else length(unique(x[!is.na(x)])) < 2

  drop_const <- covars[sapply(covars, function(v) is_constant(df_group[[v]]))]
  if (length(drop_const) > 0) {
    message("  Dropping constant covars: ", paste(drop_const, collapse = ", "))
    covars <- setdiff(covars, drop_const)
  }

  # Center year
  if ("year" %in% covars) {
    df_group  <- df_group  %>% mutate(year = year - min(year, na.rm = TRUE))
    df_before <- df_before %>% mutate(year = year - min(year, na.rm = TRUE))
  }

  form <- as.formula(paste("treated ~", paste(covars, collapse = " + ")))

  # ---------------------------------------------------------------------------
  # 3. OUTPUT DIRECTORY
  # ---------------------------------------------------------------------------

  match_mode_prefix <- if (only_fact_vars) "fact_match_" else "match_"
  output_dir <- file.path(output_dir, paste0(match_mode_prefix, match_mode))
  dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

  # ---------------------------------------------------------------------------
  # 4. MATCHING
  # ---------------------------------------------------------------------------

  use_gower <- (match_mode == "gower") && (!only_fact_vars)

  matchit_obj <- if (use_gower) {
    run_gower_matching(
      df_group, covars, demo_covars, group_col, form,
      gower_demo_weight, gower_fact_weight, caliper_q
    )
  } else if (match_mode %in% c("logit", "custom")) {
    matchit(form, data = df_group,
            method = "nearest", distance = "logit",
            caliper = caliper_ps_sd, std.caliper = TRUE,
            replace = FALSE, ratio = 1)
  } else if (match_mode == "no_caliper") {
    matchit(form, data = df_group,
            method = "nearest", distance = "logit",
            std.caliper = TRUE, replace = FALSE, ratio = 1)
  } else {
    # exact_cem: nearest neighbor + exact matching + Mahalanobis on amount
    matchit(form, data = df_group,
            method = "nearest", distance = "logit",
            exact   = c("laboratory", "self_use", "sold_to_agent", "use_offence"),
            mahvars = ~ total_amount,
            caliper = c(caliper_ps_sd, total_amount = 0.2),
            std.caliper = c(TRUE, TRUE),
            replace = FALSE, ratio = 1)
  }

  # ---------------------------------------------------------------------------
  # 5. EXTRACT MATCHED DATA
  # ---------------------------------------------------------------------------

  # Propensity scores for every unit (before matching) — used for the SM overlap figures
  if (is.numeric(matchit_obj$distance) && is.null(dim(matchit_obj$distance))) {
    write.csv(data.frame(treated  = as.integer(matchit_obj$treat),
                         distance = matchit_obj$distance,
                         matched  = as.integer(matchit_obj$weights > 0)),
              file.path(output_dir, "ps_all.csv"), row.names = FALSE)
  }

  matched_long <- get_matches(matchit_obj, data = df_group)
  if (!nrow(matched_long)) return(NULL)

  matched_long <- matched_long %>%
    add_pair_id() %>%
    arrange(pair_id, desc(treated)) %>%
    mutate(total_amount = exp(total_amount) - 0.001)

  write.csv(matched_long, file.path(output_dir, "matched_long.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)

  matched_df <- match.data(matchit_obj)
  if (!nrow(matched_df)) return(NULL)

  matched_df <- matched_df %>%
    add_pair_id() %>%
    mutate(total_amount = exp(total_amount) - 0.001)

  write.csv(matched_df, file.path(output_dir, "matched.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)

  # Re-log for modeling
  matched_df <- matched_df %>% mutate(total_amount = log(total_amount + 0.001))

  # ---------------------------------------------------------------------------
  # 6. BALANCE TABLE
  # ---------------------------------------------------------------------------

  balance <- tryCatch({
    s <- summary(matchit_obj)
    merge(
      tibble::rownames_to_column(as.data.frame(s$sum.all),     "Covariate"),
      tibble::rownames_to_column(as.data.frame(s$sum.matched), "Covariate"),
      by = "Covariate", suffixes = c("_before", "_after")
    )
  }, error = function(e) {
    message("Balance summary failed: ", conditionMessage(e))
    NULL
  })

  if (!is.null(balance))
    write.csv(balance, file.path(output_dir, "balance_table.csv"),
              fileEncoding = "UTF-8", row.names = FALSE)

  # ---------------------------------------------------------------------------
  # 7. PROPENSITY SCORE MODEL (coefficients for reporting)
  # ---------------------------------------------------------------------------

  ps_coef_df <- fit_propensity_model(df_group, covars, output_dir)

  # ---------------------------------------------------------------------------
  # 8. OUTCOME ANALYSIS
  # ---------------------------------------------------------------------------

  result_df <- compute_outcome_results(
    matched_df, matched_long, targets_cols, binary_targets, use_sensitivity = TRUE, bonferroni_k = bonferroni_k
  )
  write.csv(result_df, file.path(output_dir, "results.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)

  result_before_df <- compute_outcome_results_before(
    df_before, targets_cols, binary_targets
  )
  write.csv(result_before_df, file.path(output_dir, "results_before_matching.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)

  # ---------------------------------------------------------------------------
  # 9. PLOTS
  # ---------------------------------------------------------------------------

  ps_box_plot <- plot_ps_distribution_boxplot(
    matched_df   = matched_df,
    method_label = match_mode,
    file_path    = file.path(output_dir, "ps_boxplot.png")
  )

  # ---------------------------------------------------------------------------
  # 10. FULL REGRESSION MODELS
  # ---------------------------------------------------------------------------

  full_coef_df <- fit_full_model_after(
    matched_df, targets_cols, binary_targets, covars, output_dir,
    group_col, interaction_var, war_interaction = war_interaction
  )

  full_coef_before_df <- fit_full_model_before(
    df_before, targets_cols, binary_targets, covars, output_dir,
    group_col, interaction_var
  )

  # ---------------------------------------------------------------------------
  # 11. COMPARISON TABLE & LOVE PLOT
  # ---------------------------------------------------------------------------

  comparison_formatted <- build_comparison_table(
    df_before, matched_df, targets_cols, binary_targets, output_dir
  )

  love_plot <- create_love_plot(matchit_obj, output_dir)

  # ---------------------------------------------------------------------------
  # 12. RETURN
  # ---------------------------------------------------------------------------

  list(
    matched_df       = matched_df,
    matched_long     = matched_long,
    results          = result_df,
    comparison_table = comparison_formatted,
    balance          = balance,
    love_plot        = love_plot,
    matchit_obj      = matchit_obj,
    ps_coef          = ps_coef_df,
    df_before        = df_before,
    full_coef_after  = full_coef_df,
    full_coef_before = full_coef_before_df,
    ps_box_plot      = ps_box_plot
  )
}


# =============================================================================
# SUB-FUNCTIONS
# =============================================================================

# -----------------------------------------------------------------------------
# Gower Distance Matching
# -----------------------------------------------------------------------------
run_gower_matching <- function(df_group, covars, demo_covars, group_col,
                               form, gower_demo_weight, gower_fact_weight,
                               caliper_q) {

  X           <- df_group[, covars, drop = FALSE]
  demo_covars <- intersect(demo_covars, covars)


  # Higher weight for factual covars, lower for demographic
  w            <- setNames(rep(gower_fact_weight, length(covars)), covars)
  w[demo_covars] <- gower_demo_weight

  # Binary numerics → factors for Gower distance
  binary_vars <- names(X)[sapply(X, function(x) is.numeric(x) && length(unique(x)) == 2)]
  for (v in binary_vars) X[[v]] <- factor(X[[v]], levels = c(0, 1))

  types <- list(nominal = names(X)[sapply(X, is.factor)])
  D     <- as.matrix(daisy(X, metric = "gower", weights = w, type = types))

  idx_t    <- which(df_group$treated == 1)
  idx_c    <- which(df_group$treated == 0)
  dist_mat <- D[idx_t, idx_c, drop = FALSE]

  # finite_d   <- dist_mat[is.finite(dist_mat)]
  # caliper_val <- as.numeric(quantile(finite_d, probs = caliper_q, na.rm = TRUE))

  nn_distances <- apply(dist_mat, 1, function(r) {
    r <- r[is.finite(r)]
    if (length(r) == 0) NA_real_ else min(r)
  })
  caliper_val  <- unname(quantile(nn_distances, probs = caliper_q, na.rm = TRUE))

  matchit(form, data = df_group,
          method = "nearest", distance = dist_mat,
          caliper = caliper_val, std.caliper = FALSE,
          replace = FALSE, ratio = 1)
}


# -----------------------------------------------------------------------------
# Propensity Score Model (coefficients for reporting)
# -----------------------------------------------------------------------------
fit_propensity_model <- function(df_group, covars, output_dir) {

  ps_df         <- df_group[, c("treated", covars), drop = FALSE]
  ps_df$treated <- case_when(
    is.logical(ps_df$treated) ~ as.integer(ps_df$treated),
    is.factor(ps_df$treated)  ~ as.integer(ps_df$treated) - 1,
    TRUE                      ~ as.numeric(ps_df$treated)
  )
  ps_df <- apply_gelman_scaling(ps_df, covars)

  ps_fit <- tryCatch(
    glm(treated ~ ., data = ps_df, family = binomial()),
    error = function(e) NULL
  )
  if (is.null(ps_fit)) return(NULL)

  ps_coef_df <- as.data.frame(summary(ps_fit)$coefficients) %>%
    tibble::rownames_to_column("Covariate") %>%
    dplyr::rename(Std_Error = `Std. Error`, z_value = `z value`, p_value = `Pr(>|z|)`) %>%
    dplyr::mutate(
      Significance = cut(p_value,
                         breaks = c(-Inf, 0.001, 0.01, 0.05, Inf),
                         labels = c("***", "**", "*", ""))
    ) %>%
    arrange(p_value)

  write.csv(ps_coef_df, file.path(output_dir, "model_coefficients.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)
  ps_coef_df
}


# -----------------------------------------------------------------------------
# Outcome Analysis — After Matching
# -----------------------------------------------------------------------------
compute_outcome_results <- function(matched_df, matched_long, targets_cols,
                                    binary_targets, use_sensitivity = TRUE, bonferroni_k = 1) {
  bind_rows(lapply(targets_cols, function(col) {

    is_binary  <- col %in% binary_targets
    model_form <- as.formula(paste(col, "~ treated"))
    fit        <- fit_model(model_form, matched_df, is_binary)
    rmse_val   <- if (!is_binary) sqrt(mean(residuals(fit)^2, na.rm = TRUE)) else NA_real_

    robust_coef <- coeftest(fit, vcov = vcovHC, type = "HC3")
    ci_robust   <- lmtest::coefci(fit, vcov = vcovHC, type = "HC3", level = 0.95)
    or          <- extract_or(is_binary, robust_coef, ci_robust)

    means <- matched_df %>%
      group_by(treated) %>%
      summarise(w_mean = weighted.mean(.data[[col]], weights, na.rm = TRUE), .groups = "drop")

    x_t     <- matched_df[[col]][matched_df$treated == 1]
    x_c     <- matched_df[[col]][matched_df$treated == 0]
    d_value <- if (is_binary) {
      2 * asin(sqrt(mean(x_t, na.rm = TRUE))) - 2 * asin(sqrt(mean(x_c, na.rm = TRUE)))
    } else {
      compute_cohens_d(x_t, x_c)
    }

    gamma <- if (use_sensitivity) compute_rosenbaum_bounds(matched_long, col, is_binary)
             else NA_real_

    ev <- compute_evalues(robust_coef, ci_robust, matched_df, col,
                          is_binary, or$val, or$lo, or$hi)

    if (is_binary) {
      ame <- avg_comparisons(fit, variables = "treated",
                            vcov = sandwich::vcovHC(fit, type = "HC3"),
                            wts = matched_df$weights)
      diff_val   <- ame$estimate
      se_val     <- ame$std.error
      ci_low_val <- ame$conf.low
      ci_high_val<- ame$conf.high
      p_val      <- ame$p.value
    } else {
      diff_val   <- robust_coef["treated", 1]
      se_val     <- robust_coef["treated", 2]
      ci_low_val <- ci_robust["treated", 1]
      ci_high_val<- ci_robust["treated", 2]
      p_val      <- robust_coef["treated", 4]
    }

    tibble(
      outcome            = col,
      is_binary          = is_binary,
      mean_treatment     = means$w_mean[means$treated == 1],
      mean_control       = means$w_mean[means$treated == 0],
      diff               = diff_val,
      odds_ratio         = or$val,
      or_ci_low          = or$lo,
      or_ci_high         = or$hi,
      std_error          = se_val,
      ci_low             = ci_low_val,
      ci_high            = ci_high_val,
      d_cohens           = d_value,
      p_value            = p_val,
      p_bonferroni       = pmin(p_val * bonferroni_k, 1),
      rosenbaum_gamma    = ifelse(gamma == ">12", Inf, as.numeric(gamma)),
      e_value_point      = ev$point,
      eval_ci            = ev$ci,
      n_treatment        = sum(matched_df$weights[matched_df$treated == 1]),
      n_control          = sum(matched_df$weights[matched_df$treated == 0]),
      n_treatment_unique = sum(matched_df$treated == 1),
      n_control_unique   = sum(matched_df$treated == 0),
      sd_treatment       = sd(x_t, na.rm = TRUE),
      sd_control         = sd(x_c, na.rm = TRUE),
      rmse               = rmse_val
    )
  }))
}


# -----------------------------------------------------------------------------
# Rosenbaum Bounds
# -----------------------------------------------------------------------------
compute_rosenbaum_bounds <- function(matched_long, col, is_binary) {

  pairs_data <- matched_long %>%
    select(pair_id, treated, all_of(col)) %>%
    group_by(pair_id, treated) %>%
    summarise(across(all_of(col), first), .groups = "drop") %>%
    pivot_wider(names_from = treated, values_from = all_of(col),
                names_prefix = "treat_") %>%
    drop_na()

  if (nrow(pairs_data) <= 10) return(NA_real_)

  gamma_tipping_point <- NA_real_
  pdf(file = NULL)
  on.exit(dev.off(), add = TRUE)
  try({
    rb_results <- if (is_binary) {
      n_plus  <- sum(pairs_data$treat_1 == 1 & pairs_data$treat_0 == 0)
      n_minus <- sum(pairs_data$treat_1 == 0 & pairs_data$treat_0 == 1)
      rbounds::binarysens(n_plus, n_minus, Gamma = 12, GammaInc = 0.1)
    } else {
      rbounds::psens(pairs_data$treat_1, pairs_data$treat_0, Gamma = 12, GammaInc = 0.1)
    }

    bounds_df   <- rb_results$bounds
    tipping_row <- which(bounds_df[, "Upper bound"] > 0.05)[1]
    gamma_tipping_point <- if (!is.na(tipping_row)) bounds_df[tipping_row, "Gamma"] else ">12"
  }, silent = TRUE)

  gamma_tipping_point
}


# -----------------------------------------------------------------------------
# E-Values
# -----------------------------------------------------------------------------
compute_evalues <- function(robust_coef, ci_robust, matched_df, col,
                            is_binary, or_val, or_ci_low, or_ci_high) {

  eval_point <- NA_real_
  eval_ci    <- NA_real_

  suppressMessages(try({
    ev_obj <- if (is_binary) {
      EValue::evalues.OR(est = or_val, lo = or_ci_low, hi = or_ci_high, rare = FALSE)
    } else {
      EValue::evalues.OLS(
        est = robust_coef["treated", 1],
        se  = robust_coef["treated", 2],
        sd  = sd(matched_df[[col]], na.rm = TRUE)
      )
    }
    eval_point <- ev_obj[2, "point"]
    eval_ci    <- ev_obj[2, "lower"]
  }, silent = TRUE))

  list(point = eval_point, ci = eval_ci)
}


# -----------------------------------------------------------------------------
# Outcome Analysis — Before Matching
# -----------------------------------------------------------------------------
compute_outcome_results_before <- function(df_before, targets_cols, binary_targets) {

  bind_rows(lapply(targets_cols, function(col) {

    is_binary   <- col %in% binary_targets
    model_form  <- as.formula(paste(col, "~ treated"))
    fit         <- fit_model(model_form, df_before, is_binary)
    rmse_val    <- if (!is_binary) sqrt(mean(residuals(fit)^2, na.rm = TRUE)) else NA_real_

    robust_coef <- coeftest(fit, vcov = vcovHC, type = "HC3")
    ci_robust   <- lmtest::coefci(fit, vcov = vcovHC, type = "HC3", level = 0.95)
    or          <- extract_or(is_binary, robust_coef, ci_robust)

    means <- df_before %>%
      group_by(treated) %>%
      summarise(mean = mean(.data[[col]], na.rm = TRUE), .groups = "drop")

    x_t <- df_before[[col]][df_before$treated == 1]
    x_c <- df_before[[col]][df_before$treated == 0]

    tibble(
      outcome        = col,
      is_binary      = is_binary,
      mean_treatment = means$mean[means$treated == 1],
      mean_control   = means$mean[means$treated == 0],
      diff           = robust_coef["treated", 1],
      odds_ratio     = or$val,
      or_ci_low      = or$lo,
      or_ci_high     = or$hi,
      std_error      = robust_coef["treated", 2],
      ci_low         = ci_robust["treated", 1],
      ci_high        = ci_robust["treated", 2],
      d_cohens       = compute_cohens_d(x_t, x_c),
      p_value        = robust_coef["treated", 4],
      sd_treatment   = sd(x_t, na.rm = TRUE),
      sd_control     = sd(x_c, na.rm = TRUE),
      n_treatment    = sum(!is.na(x_t)),
      n_control      = sum(!is.na(x_c)),
      rmse           = rmse_val
    )
  }))
}


# -----------------------------------------------------------------------------
# Full Regression Model — After Matching (cluster-robust SE)
# -----------------------------------------------------------------------------
fit_full_model_after <- function(matched_df, targets_cols, binary_targets,
                                 covars, output_dir, group_col, interaction_var,
                                 war_interaction = FALSE) {   # <<< חדש

  full_coef_df <- bind_rows(lapply(targets_cols, function(col) {

    valid_covars <- Filter(function(v) {
      if (!v %in% names(matched_df)) return(FALSE)
      x <- matched_df[[v]]
      if (is.factor(x) || is.character(x)) nlevels(droplevels(as.factor(x))) >= 2
      else var(x, na.rm = TRUE) > 0
    }, covars)

    covars_for_model <- if (group_col == "ethnic") {
      valid_covars[valid_covars != interaction_var]
    } else {
      valid_covars
    }

    # Only build the war_period interaction when explicitly requested,
    # the column exists, AND it isn't collinear with `treated`.
    use_war_ix <- war_interaction &&
                  "war_period" %in% names(matched_df) &&
                  !identical(matched_df$war_period, matched_df$treated)

    rhs <- if (length(covars_for_model) == 0) {
      "treated"
    } else if (use_war_ix) {
      paste0("treated * war_period + ",
             paste(setdiff(covars_for_model, "war_period"), collapse = " + "))
    } else {
      paste0("treated + ", paste(covars_for_model, collapse = " + "))
    }

    model_form <- as.formula(paste(col, "~", rhs))
    is_binary  <- col %in% binary_targets
    fit <- tryCatch(fit_model(model_form, matched_df, is_binary), error = function(e) NULL)
    if (is.null(fit)) return(NULL)

    extract_model_coefs(fit, matched_df, col, is_binary, cluster_var = "pair_id")
  }))

  write.csv(full_coef_df, file.path(output_dir, "full_model_coefficients.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)
  full_coef_df
}


# -----------------------------------------------------------------------------
# Full Regression Model — Before Matching (HC3 robust SE)
# -----------------------------------------------------------------------------
fit_full_model_before <- function(df_before, targets_cols, binary_targets,
                                  covars, output_dir, group_col, interaction_var) {

  full_coef_df <- bind_rows(lapply(targets_cols, function(col) {

    is_binary <- col %in% binary_targets

    covars_for_model <- if (group_col == "ethnic") {
      covars[covars != interaction_var]
    } else {
      covars
    }

    model_form <- as.formula(
      paste(col, "~ treated +", paste(covars_for_model, collapse = " + "))
    )
    fit <- fit_model(model_form, df_before, is_binary)

    robust_vcov <- tryCatch({
      vc <- vcovHC(fit, type = "HC3")
      if (any(is.na(diag(vc)))) vcov(fit) else vc
    }, error = function(e) vcov(fit))

    extract_model_coefs_from_vcov(fit, robust_vcov, col, is_binary)
  }))

  write.csv(full_coef_df, file.path(output_dir, "full_model_coefficients_before.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)
  full_coef_df
}


# -----------------------------------------------------------------------------
# Extract Model Coefficients (shared helpers)
# -----------------------------------------------------------------------------
extract_model_coefs <- function(fit, data, outcome_col, is_binary,
                                cluster_var = NULL) {
  robust_vcov <- if (!is.null(cluster_var)) vcovCL(fit, cluster = data[[cluster_var]])
                 else vcovHC(fit, type = "HC3")

  extract_model_coefs_from_vcov(fit, robust_vcov, outcome_col, is_binary)
}

extract_model_coefs_from_vcov <- function(fit, robust_vcov, outcome_col, is_binary) {

  robust_coef   <- coeftest(fit, vcov = robust_vcov)
  ci_robust     <- coefci(fit, vcov. = robust_vcov, level = 0.95)
  model_summary <- summary(fit)
  rmse          <- sqrt(mean(residuals(fit)^2))

  if (is_binary) {
    r_squared     <- 1 - model_summary$deviance / model_summary$null.deviance
    adj_r_squared <- NA_real_
    f_stat        <- NA_real_
    f_p_value     <- NA_real_
  } else {
    r_squared     <- model_summary$r.squared
    adj_r_squared <- model_summary$adj.r.squared
    f_stat        <- model_summary$fstatistic[1]
    f_p_value     <- pf(f_stat,
                        model_summary$fstatistic[2],
                        model_summary$fstatistic[3],
                        lower.tail = FALSE)
  }

  data.frame(
    term    = rownames(robust_coef),
    B       = robust_coef[, 1],
    SE      = robust_coef[, 2],
    t_value = robust_coef[, 3],
    p_value = robust_coef[, 4],
    CI_low  = ci_robust[, 1],
    CI_high = ci_robust[, 2],
    RMSE    = rmse
  ) %>%
    mutate(
      outcome       = outcome_col,
      R_squared     = r_squared,
      Adj_R_squared = adj_r_squared,
      F_statistic   = f_stat,
      F_p_value     = f_p_value,
      sig           = sig_label(p_value)
    ) %>%
    select(outcome, term, B, SE, CI_low, CI_high, t_value, p_value, sig,
           R_squared, Adj_R_squared, RMSE, F_statistic, F_p_value)
}


# -----------------------------------------------------------------------------
# Comparison Table (Before vs After)
# -----------------------------------------------------------------------------
build_comparison_table <- function(df_before, matched_df, targets_cols,
                                   binary_targets, output_dir) {

  comparison_table <- bind_rows(lapply(targets_cols, function(outcome) {
    form      <- as.formula(paste(outcome, "~ treated"))
    is_binary <- outcome %in% binary_targets
    fit_before  <- fit_model(form, df_before, is_binary)
    vcov_before <- vcovHC(fit_before, type = "HC3")
    test_before <- coeftest(fit_before, vcov = vcov_before)
    ci_before   <- coefci(fit_before,  vcov. = vcov_before)
    fit_after  <- fit_model(form, matched_df, is_binary)
    vcov_after <- vcovCL(fit_after, cluster = matched_df$pair_id)
    test_after <- coeftest(fit_after, vcov = vcov_after)
    ci_after   <- coefci(fit_after,  vcov. = vcov_after)
    build_row <- function(term_name, coef_name) {
      tibble(
        outcome        = outcome,
        term           = term_name,
        B_before       = test_before[coef_name, 1],
        SE_before      = test_before[coef_name, 2],
        CI_low_before  = ci_before[coef_name,   1],
        CI_high_before = ci_before[coef_name,   2],
        p_before       = test_before[coef_name, 4],
        B_after        = test_after[coef_name,  1],
        SE_after       = test_after[coef_name,  2],
        CI_low_after   = ci_after[coef_name,    1],
        CI_high_after  = ci_after[coef_name,    2],
        p_after        = test_after[coef_name,  4]
      )
    }

    bind_rows(
      build_row("Treatment", "treated"),
      build_row("Intercept", "(Intercept)")
    )
  }))

  comparison_formatted <- comparison_table %>%
    mutate(
      N_before   = nrow(df_before),
      N_after    = nrow(matched_df),
      N_weighted = nrow(matched_df)
    ) %>%
    mutate(across(c(B_before, B_after, SE_before, SE_after,
                    CI_low_before, CI_high_before, CI_low_after, CI_high_after),
                  ~ round(., 3))) %>%
    mutate(
      CI_before    = paste0(CI_low_before,  ", ", CI_high_before),
      CI_after     = paste0(CI_low_after,   ", ", CI_high_after),
      B_before_fmt = paste0(B_before, sig_label(p_before)),
      B_after_fmt  = paste0(B_after,  sig_label(p_after))
    ) %>%
    select(outcome, term,
           B_before_fmt, SE_before, CI_before,
           B_after_fmt,  SE_after,  CI_after,
           N_before, N_after, N_weighted)

  write.csv(comparison_formatted, file.path(output_dir, "comparison_table.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)
  comparison_formatted
}


# -----------------------------------------------------------------------------
# Love Plot
# -----------------------------------------------------------------------------
create_love_plot <- function(matchit_obj, output_dir) {
  library(cobalt)
  library(ggplot2)

  COL_BEFORE  <- "#E64B35"
  COL_AFTER   <- "#3C5488"
  COL_GRID    <- "#F0F0F0"
  COL_TEXT    <- "#000000"
  FONT_FAMILY <- "sans"

  love_plot <- tryCatch({
    love.plot(
      matchit_obj,
      abs           = TRUE,
      thresholds    = 0.25,
      drop.distance = TRUE,
      var.order     = "unadjusted",
      sample.names  = c("Before Matching", "After Matching"),
      colors        = c(COL_BEFORE, COL_AFTER),
      shapes        = c(16, 16),
      stats         = "mean.diffs",
      binary        = "std",
      size          = 3.5,
      line          = FALSE
    )
  }, error = function(e) {
    message("love.plot failed: ", conditionMessage(e))
    NULL
  })

  if (is.null(love_plot)) return(invisible(NULL))

  polished_plot <- love_plot +
    labs(x = "Absolute Standardized Mean Difference", y = NULL,
         title = NULL, subtitle = NULL, colour = NULL, shape = NULL) +
    scale_y_discrete(labels = love_var_translate) +
    theme_minimal(base_size = 11, base_family = FONT_FAMILY) +
    theme(
      panel.grid.major.x  = element_line(colour = COL_GRID, linewidth = 0.5),
      panel.grid.minor.x  = element_blank(),
      panel.grid.major.y  = element_line(colour = COL_GRID, linewidth = 0.5),
      axis.text.y         = element_text(size = 10, colour = COL_TEXT, hjust = 1),
      axis.text.x         = element_text(size = 10, colour = COL_TEXT),
      axis.title.x        = element_text(size = 11, colour = COL_TEXT, margin = margin(t = 10)),
      legend.position     = "bottom",
      legend.justification = "center",
      legend.box          = "horizontal",
      legend.box.just     = "center",
      legend.background   = element_blank(),
      legend.key          = element_blank(),
      legend.text         = element_text(size = 10, colour = COL_TEXT),
      plot.margin         = margin(12, 12, 12, 12)
    ) +
    guides(
      colour = guide_legend(override.aes = list(size = 5.5)),
      shape  = guide_legend(override.aes = list(size = 5.5))
    )

  num_vars           <- length(unique(love_plot$data$var))
  calculated_height  <- max(4, min(11, 2 + num_vars * 0.25))

  dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
  ggsave(file.path(output_dir, "love_plot.png"),
         plot = polished_plot, width = 9.5, height = calculated_height,
         dpi = 300, bg = "white")

  invisible(love_plot)
}

In [ ]:
# =============================================================================
# SENSITIVITY KINDS (SM, Section 3.3)
#   impute / impute_0 / impute_6 / no_impute : data read from data/<kind>/
#   no_personal : primary imputed data, without age / marital status / criminal record
#   ps_overlap  : primary data and covariates again, to export full-sample propensity scores (ps_all.csv)
#   unfiltered  : includes cases without a declared sentencing range (incarceration only)
# =============================================================================
PERSONAL_COVARS <- c("age", "marital_status", "Criminal_record")

data_dir_for <- function(kind) if (kind %in% c("no_personal", "ps_overlap")) "impute" else kind

drop_personal_covars <- function(cv, kind) {
  if (kind != "no_personal") return(cv)
  lapply(cv, setdiff, PERSONAL_COVARS)
}

targets_for <- function(kind) {
  if (kind == "unfiltered") return("is_actual_sentence")
  c("min_punishment", "max_punishment", "is_actual_sentence")
}

# Analysis from csv

In [ ]:
analyze_pairs_from_csv <- function(
    df,
    pairs_path,
    output_dir,
    group_col,
    id_col,
    treatment,
    control,
    targets_cols,
    all_covars
) {
  
  # =========================
  # Helper Functions
  # =========================
  
  compute_cohens_d <- function(x_t, x_c) {
    n_t <- sum(!is.na(x_t))
    n_c <- sum(!is.na(x_c))
    if (n_t < 2 || n_c < 2) return(NA_real_)
    
    sd_pooled <- sqrt(((n_t - 1) * var(x_t, na.rm = TRUE) + 
                         (n_c - 1) * var(x_c, na.rm = TRUE)) / (n_t + n_c - 2))
    if (!is.finite(sd_pooled) || sd_pooled == 0) return(NA_real_)
    
    (mean(x_t, na.rm = TRUE) - mean(x_c, na.rm = TRUE)) / sd_pooled
  }
  
  smd_numeric <- function(x, t) {
    t_vals <- x[t == 1]
    c_vals <- x[t == 0]
    
    if (length(unique(na.omit(t_vals))) < 2 && length(unique(na.omit(c_vals))) < 2) {
      return(NA_real_)
    }
    
    vt <- var(t_vals, na.rm = TRUE)
    vc <- var(c_vals, na.rm = TRUE)
    
    if (!is.finite(vt) || !is.finite(vc) || (vt + vc) == 0) return(NA_real_)
    
    abs((mean(t_vals, na.rm = TRUE) - mean(c_vals, na.rm = TRUE)) / sqrt((vt + vc) / 2))
  }
  
  smd_factor <- function(x, t, varname) {
    levs <- levels(x)
    bind_rows(lapply(levs[-1], function(lv) {
      tibble(
        variable = paste0(varname, " = ", lv),
        smd = smd_numeric(as.numeric(x == lv), t)
      )
    }))
  }
  
  # =========================
  # Setup
  # =========================
  
  dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
  
  # =========================
  # Covariates Setup
  # =========================
  
  
  covars <- c(setdiff(all_covars, excluded))
  
  # =========================
  # Data Preprocessing
  # =========================
  
  df_before <- df %>%
    filter(.data[[group_col]] %in% c(treatment, control)) %>%
    drop_na(any_of(c(group_col, covars, targets_cols))) %>%
    mutate(
      treated = as.integer(.data[[group_col]] == treatment),
      verdict_id = as.character(.data[[id_col]])
    )
  
  df_group <- df_before
  
  # Log transform for continuous amount
  if ("total_amount" %in% names(df_group)) {
    df_group <- df_group %>%
      mutate(total_amount = log(total_amount + 0.001))
  }
  
  # Factorize categorical variables
  for (v in covars) {
    if (v %in% names(df_group)) {
      if (is.character(df_group[[v]]) || dplyr::n_distinct(df_group[[v]]) < 10) {
        df_group[[v]] <- as.factor(df_group[[v]])
      }
    }
  }


  # =========================
  # Reconstruct Matched Pairs
  # =========================
  
  pairs <- read_csv(pairs_path, show_col_types = FALSE) %>%
    mutate(
      pair_id = row_number(),
      across(c(id_treat, id_control), as.character)
    )
  
  pairs_long_map <- pairs %>%
    tidyr::pivot_longer(
      cols = c(id_treat, id_control),
      names_to = "type",
      values_to = "verdict_id"
    ) %>%
    mutate(treated = ifelse(type == "id_treat", 1L, 0L))
  
  matched_pairs_long <- df_group %>%
    inner_join(pairs_long_map, by = c("verdict_id", "treated")) %>%
    arrange(pair_id, desc(treated))

  # =========================
  # Calculate SMD (Before/After)
  # =========================
  
  smd_before <- bind_rows(lapply(covars, function(v) {
    x <- df_group[[v]]
    if (is.factor(x)) {
      smd_factor(x, df_group$treated, v) %>% mutate(time = "Before")
    } else {
      tibble(variable = v, smd = smd_numeric(x, df_group$treated), time = "Before")
    }
  }))
  
  smd_after <- bind_rows(lapply(covars, function(v) {
    x <- matched_pairs_long[[v]]
    if (is.factor(x)) {
      smd_factor(x, matched_pairs_long$treated, v) %>% mutate(time = "After")
    } else {
      tibble(variable = v, smd = smd_numeric(x, matched_pairs_long$treated), time = "After")
    }
  }))
  
  smd_long <- bind_rows(smd_before, smd_after) %>%
    filter(!is.na(smd))
  
  write_csv(smd_long, file.path(output_dir, "SMD.csv"))
  
  # =========================
  # Love Plot
  # =========================
  
  order_vars <- smd_long %>%
    filter(time == "Before") %>%
    arrange(smd) %>%
    pull(variable)
  
  smd_long$variable <- factor(smd_long$variable, levels = unique(order_vars))
  
  love_plot <- ggplot(smd_long, aes(x = smd, y = variable, color = time)) +
    geom_point(size = 2) +
    geom_vline(xintercept = 0.25, linetype = "dashed", alpha = 0.6) +
    scale_color_manual(values = c(Before = "#e11d48", After = "#0284c7")) +
    labs(title = "Covariate Balance", x = "|SMD|", y = NULL) +
    theme_minimal(base_size = 14) +
    theme(
      plot.background = element_rect(fill = "white", color = NA),
      panel.background = element_rect(fill = "white", color = NA),
      legend.background = element_rect(fill = "white", color = NA),
      legend.key = element_rect(fill = "white", color = NA)
    )
  
  ggsave(
    file.path(output_dir, "love_plot.png"),
    love_plot,
    width = 10,
    height = 7,
    dpi = 300,
    bg = "white"
  )
  
  # =========================
  # Outcome Analysis (Paired t-test)
  # =========================
  
  results_df <- bind_rows(lapply(targets_cols, function(outcome) {
    
    paired_data <- matched_pairs_long %>%
      select(pair_id, treated, !!sym(outcome)) %>%
      filter(is.finite(!!sym(outcome))) %>%
      group_by(pair_id) %>%
      filter(n() == 2) %>%
      arrange(pair_id, desc(treated)) %>%
      ungroup()
    
    if (nrow(paired_data) < 4) return(NULL)
    
    t_vals <- paired_data %>% filter(treated == 1) %>% pull(outcome)
    c_vals <- paired_data %>% filter(treated == 0) %>% pull(outcome)
    
    tt <- t.test(t_vals, c_vals, paired = TRUE)
    
    # OR via conditional logistic regression (only for binary outcomes)
    or_val <- NA_real_
    or_ci_low <- NA_real_
    or_ci_high <- NA_real_
    
    if (all(paired_data[[outcome]] %in% c(0, 1, NA))) {
      tryCatch({
        fit <- survival::clogit(
          as.formula(paste(outcome, "~ treated + strata(pair_id)")),
          data = paired_data
        )
        ci_vals <- confint(fit)["treated", ]
        or_val     <- exp(coef(fit)["treated"])
        or_ci_low  <- exp(ci_vals[1])
        or_ci_high <- exp(ci_vals[2])
      }, error = function(e) NULL)
    }
    
    tibble(
      outcome        = outcome,
      mean_treatment = mean(t_vals),
      mean_control   = mean(c_vals),
      diff           = mean(t_vals) - mean(c_vals),
      std_error      = tt$stderr,
      ci_low         = tt$conf.int[1],
      ci_high        = tt$conf.int[2],
      d_cohens       = compute_cohens_d(t_vals, c_vals),
      p_value        = tt$p.value,
      n_pairs        = length(t_vals),
      n_treatment    = sum(!is.na(t_vals)),
      n_control      = sum(!is.na(c_vals)),
      sd_treatment   = sd(t_vals),
      sd_control     = sd(c_vals),
      OR             = or_val,
      OR_ci_low      = or_ci_low,
      OR_ci_high     = or_ci_high
    )
  }))

  
  write_csv(results_df, file.path(output_dir, "results.csv"))
  
  # =========================
  # Comparison Table (Before vs After)
  # =========================
  
  comparison_table <- bind_rows(lapply(targets_cols, function(outcome) {
    
    form <- as.formula(paste(outcome, "~ treated"))
    
    # --- Before Matching (OLS) ---
    fit_before <- lm(form, data = df_before)
    coef_before <- summary(fit_before)$coefficients
    ci_before <- confint(fit_before)
    
    # --- After Matching (Paired data as OLS) ---
    paired_data <- matched_pairs_long %>%
      select(pair_id, treated, !!sym(outcome)) %>%
      filter(is.finite(!!sym(outcome))) %>%
      group_by(pair_id) %>%
      filter(n() == 2) %>%
      ungroup()
    
    if (nrow(paired_data) < 4) return(NULL)
    
    fit_after <- lm(form, data = paired_data)
    coef_after <- summary(fit_after)$coefficients
    ci_after <- confint(fit_after)
    
    treatment_row <- tibble(
      outcome = outcome,
      term = "Treatment",
      
      B_before = coef_before["treated", "Estimate"],
      SE_before = coef_before["treated", "Std. Error"],
      CI_low_before = ci_before["treated", 1],
      CI_high_before = ci_before["treated", 2],
      p_before = coef_before["treated", "Pr(>|t|)"],
      
      B_after = coef_after["treated", "Estimate"],
      SE_after = coef_after["treated", "Std. Error"],
      CI_low_after = ci_after["treated", 1],
      CI_high_after = ci_after["treated", 2],
      p_after = coef_after["treated", "Pr(>|t|)"]
    )
    
    intercept_row <- tibble(
      outcome = outcome,
      term = "Intercept",
      
      B_before = coef_before["(Intercept)", "Estimate"],
      SE_before = coef_before["(Intercept)", "Std. Error"],
      CI_low_before = ci_before["(Intercept)", 1],
      CI_high_before = ci_before["(Intercept)", 2],
      p_before = coef_before["(Intercept)", "Pr(>|t|)"],
      
      B_after = coef_after["(Intercept)", "Estimate"],
      SE_after = coef_after["(Intercept)", "Std. Error"],
      CI_low_after = ci_after["(Intercept)", 1],
      CI_high_after = ci_after["(Intercept)", 2],
      p_after = coef_after["(Intercept)", "Pr(>|t|)"]
    )
    
    bind_rows(treatment_row, intercept_row)
  }))
  
  if (!is.null(comparison_table) && nrow(comparison_table) > 0) {
    
    comparison_table <- comparison_table %>%
      mutate(
        N_before = nrow(df_before),
        N_after = nrow(matched_pairs_long),
        N_pairs = nrow(matched_pairs_long) / 2
      )
    
    comparison_formatted <- comparison_table %>%
      mutate(
        B_before = round(B_before, 3),
        B_after = round(B_after, 3),
        SE_before = round(SE_before, 3),
        SE_after = round(SE_after, 3),
        CI_low_before = round(CI_low_before, 3),
        CI_high_before = round(CI_high_before, 3),
        CI_low_after = round(CI_low_after, 3),
        CI_high_after = round(CI_high_after, 3),
        
        CI_before = paste0(CI_low_before, ", ", CI_high_before),
        CI_after = paste0(CI_low_after, ", ", CI_high_after),
        
        sig_before = case_when(
          p_before <= 0.001 ~ "***",
          p_before <= 0.01 ~ "**",
          p_before <= 0.05 ~ "*",
          TRUE ~ ""
        ),
        sig_after = case_when(
          p_after <= 0.001 ~ "***",
          p_after <= 0.01 ~ "**",
          p_after <= 0.05 ~ "*",
          TRUE ~ ""
        ),
        
        B_before_fmt = paste0(B_before, sig_before),
        B_after_fmt = paste0(B_after, sig_after)
      ) %>%
      select(
        outcome, term,
        B_before_fmt, SE_before, CI_before,
        B_after_fmt, SE_after, CI_after,
        N_before, N_after, N_pairs
      )
    
    write_csv(comparison_formatted, file.path(output_dir, "comparison_table.csv"))
    
  } else {
    comparison_formatted <- NULL
  }
  
  # =========================
  # Save Matched Data
  # =========================
  
  if ("total_amount" %in% names(matched_pairs_long)) {
    matched_pairs_long <- matched_pairs_long %>%
      mutate(total_amount = exp(total_amount) - 0.001)
  }
  
  write_csv(matched_pairs_long, file.path(output_dir, "matched.csv"))
  
  # =========================
  # Return
  # =========================
  # כמה יש לפני drop_na

  invisible(list(
    matched_pairs_long = matched_pairs_long,
    df_before          = df_before,
    smd_long           = smd_long,
    results            = results_df,
    comparison_table   = comparison_formatted,
    love_plot          = love_plot
  ))
}

# Run Defendants Race

In [ ]:
 # =============================================================================
# CONFIG
# =============================================================================
options(warn = -1)

CONFIG <- list(
  kinds       = c("no_impute","no_personal","unfiltered"),
  drugs       = c("cannabis", "cocaine", "hashish"),
  match_modes = c("logit"),
  group_col   = "ethnic",
  treatment   = "Arab",
  control     = "Jewish",
  caliper_q   = 0.9,
  caliper_ps  = 0.2,
  gower_demo_w = 1,
  gower_fact_w = 3,
  interaction_var = "war_period",
  use_punishment_months = FALSE
)

# Covariate templates — copied per-iteration so they never mutate globally
COVARS_BASE <- list(
  all = c("district","age","year","Court",
          "production_offence","use_offence","importation_offence",
          "laboratory","self_use","ethnic","sold_to_agent","total_amount",
          "regret","confession","Criminal_record","marital_status","judge_ethnic"),
  fact = c("production_offence","use_offence","importation_offence",
           "laboratory","self_use","sold_to_agent","total_amount"),
  demo = c("district","age","year","ethnic","regret","confession",
           "Court","Criminal_record","marital_status","judge_ethnic")
)

TARGETS <- list(
  main = c("min_punishment", "max_punishment", "is_actual_sentence"),
  pm   = c("punishment_months")
)
BINARY_TARGETS <- c("is_actual_sentence")

# =============================================================================
# HELPER: adjust covariates for a given (kind, drug) WITHOUT mutating globals
# =============================================================================
covars_for <- function(kind, drug) {
  cv <- COVARS_BASE
  if (drug != "cannabis") {
    cv$all  <- setdiff(cv$all,  "production_offence")
    cv$fact <- setdiff(cv$fact, "production_offence")
  }
  cv
}

# =============================================================================
# HELPER: load + clean one drug's data
# =============================================================================
load_drug_data <- function(kind, drug) {
  path <- paste0("data/", data_dir_for(kind), "/", drug, ".csv")
  read_csv(path, locale = locale(encoding = "UTF-8"), show_col_types = FALSE) %>%
    mutate(
      total_amount       = total_amount / 1000,
      district           = str_squish(as.character(district)),
      age                = as.numeric(age),
      is_actual_sentence = as.integer(is_actual_sentence)
    ) %>%
    filter(!(ethnic %in% c("Anonymous", "Unknown")))
}

# =============================================================================
# HELPER: two-pass matching (main targets + optional punishment_months)
# =============================================================================
run_matching_two_passes <- function(df, output_dir, match_mode, cv,
                                     only_fact_vars = FALSE, targets = TARGETS$main) {
  prefix        <- if (only_fact_vars) "fact_match_" else "match_"
  method_subdir <- file.path(output_dir, paste0(prefix, match_mode))
  before_path   <- file.path(output_dir, "results_before_matching.csv")
  res_path      <- file.path(method_subdir, "results.csv")

  common_args <- list(
    group_col       = CONFIG$group_col,
    output_dir      = output_dir,
    treatment       = CONFIG$treatment,
    control         = CONFIG$control,
    match_mode      = match_mode,
    caliper_q       = CONFIG$caliper_q,
    caliper_ps_sd   = CONFIG$caliper_ps,
    all_covars      = cv$all,
    fact_covars     = cv$fact,
    demo_covars     = cv$demo,
    gower_demo_weight = CONFIG$gower_demo_w,
    gower_fact_weight = CONFIG$gower_fact_w,
    only_fact_vars  = only_fact_vars,
    interaction_var = CONFIG$interaction_var,
    war_interaction = TRUE
  )

  # --- Pass 1: main targets, full data ---
  cat("  [Pass 1] Main targets on full data\n")
  tryCatch(
    do.call(run_matching, c(list(df = df, targets_cols = targets,
                                  binary_targets = BINARY_TARGETS), common_args)),
    error = function(e) cat("  Error pass 1:", conditionMessage(e), "\n")
  )

  pass1_results <- if (file.exists(res_path))    read.csv(res_path,    fileEncoding = "UTF-8") else NULL
  pass1_before  <- if (file.exists(before_path)) read.csv(before_path, fileEncoding = "UTF-8") else NULL

  if (!CONFIG$use_punishment_months) return(invisible(NULL))

  # --- Pass 2: punishment_months, filtered data ---
  df_pm <- df %>% filter(punishment_months > 0)
  cat("  [Pass 2] punishment_months (N =", nrow(df_pm), ")\n")
  tryCatch(
    do.call(run_matching, c(list(df = df_pm, targets_cols = TARGETS$pm,
                                  binary_targets = NULL), common_args)),
    error = function(e) cat("  Error pass 2:", conditionMessage(e), "\n")
  )

  # --- Merge pass 1 + pass 2 ---
  if (!is.null(pass1_results) && file.exists(res_path)) {
    write.csv(rbind(pass1_results, read.csv(res_path, fileEncoding = "UTF-8")),
              res_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
  if (!is.null(pass1_before) && file.exists(before_path)) {
    write.csv(rbind(pass1_before, read.csv(before_path, fileEncoding = "UTF-8")),
              before_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
}

# =============================================================================
# MAIN LOOP
# =============================================================================
for (kind in CONFIG$kinds) {
  for (drug in CONFIG$drugs) {
    cv      <- drop_personal_covars(covars_for(kind, drug), kind)   # local copy — no global mutation
    df      <- load_drug_data(kind, drug)
    out_dir <- paste0("matched_outputs/", kind, "/", drug, "/Defendants_Ethnic")

    cat("\n=== Processing:", kind, "/", drug, "===\n")

    for (mm in CONFIG$match_modes) {
      cat("\n--- match_mode:", mm, "---\n")
      run_matching_two_passes(df = df, output_dir = out_dir,
                              match_mode = mm, cv = cv, only_fact_vars = FALSE,
                              targets = targets_for(kind))
    }
  }
}

# # --- Pair-based analysis ---
# cat("\n--- Analyzing pairs from CSV ---\n")
# tryCatch(
#   analyze_pairs_from_csv(
#     df           = df,
#     pairs_path   = paste0("data/pairs/", CONFIG$group_col, "/", drug, ".csv"),
#     output_dir   = file.path(out_dir, "manual_pairs"),
#     group_col    = CONFIG$group_col,
#     id_col       = "original_case",
#     treatment    = CONFIG$treatment,
#     control      = CONFIG$control,
#     targets_cols = TARGETS$main,
#     all_covars   = cv$all
#   ),
#   error = function(e) cat("  Error in pair analysis:", conditionMessage(e), "\n")
#     )

# District

In [ ]:
# =============================================================================
# DISTRICT ANALYSIS — ONE VS ALL COMPARISONS
# =============================================================================

CONFIG <- list(
  kinds       = c("no_impute","no_personal"),
  drugs         = c("cannabis", "cocaine", "hashish"),
  match_modes   = c("logit"),
  caliper_q     = 0.9,
  caliper_ps    = 0.2,
  gower_demo_w  = 1,
  gower_fact_w  = 3,
  min_district_n = 10,          # min obs per district to include
  min_group_n    = 10,          # min obs per treatment/control arm
  control_label  = "All_Others",
  group_col      = "district_one_vs_all",
  use_punishment_months = FALSE
)

TARGETS <- list(
  main = c("min_punishment", "max_punishment", "is_actual_sentence"),
  pm   = c("punishment_months")
)
BINARY_TARGETS <- c("is_actual_sentence")

# =============================================================================
# HELPER: covariate sets for a given (kind, drug) — local copy, no mutation
# =============================================================================
covars_for <- function(kind, drug) {
  cv <- list(
    all = c("age","marital_status","year","Court",
            "production_offence","use_offence","importation_offence",
            "laboratory","self_use","ethnic","sold_to_agent","total_amount",
            "regret","confession","Criminal_record","judge_ethnic"),
    fact = c("production_offence","use_offence","importation_offence",
             "laboratory","self_use","sold_to_agent","total_amount","judge_ethnic"),
    demo = c("age","marital_status","year","ethnic",
             "regret","confession","Court","Criminal_record")
  )
  if (kind == "only_age") {
    cv$all  <- setdiff(cv$all,  "marital_status")
    cv$demo <- setdiff(cv$demo, "marital_status")
  }
  if (drug != "cannabis") {
    cv$all  <- setdiff(cv$all,  "production_offence")
    cv$fact <- setdiff(cv$fact, "production_offence")
  }
  cv
}

# =============================================================================
# HELPER: load + clean one drug's data
# =============================================================================
load_drug_data <- function(kind, drug) {
  path <- paste0("data/", data_dir_for(kind), "/", drug, ".csv")
  read_csv(path, locale = locale(encoding = "UTF-8"), show_col_types = FALSE) %>%
    filter(Criminal_record != "לא ידוע") %>%
    mutate(
      total_amount       = total_amount / 1000,
      district           = str_squish(as.character(district)),
      age                = as.numeric(age),
      is_actual_sentence = as.integer(is_actual_sentence)
    ) %>%
    filter(!(ethnic %in% c("Anonymous", "Unknown"))) %>%
    filter(!is.na(district) & district != "")
}

# =============================================================================
# HELPER: districts with enough observations
# =============================================================================
eligible_districts <- function(df) {
  df %>%
    count(district, name = "n") %>%
    filter(n >= CONFIG$min_district_n) %>%
    arrange(district) %>%
    pull(district)
}

# =============================================================================
# HELPER: two-pass matching (main + optional punishment_months)
# =============================================================================
run_matching_two_passes <- function(df, output_dir, treatment, control,
                                     match_mode, cv, only_fact_vars = FALSE) {
  prefix        <- if (only_fact_vars) "fact_match_" else "match_"
  method_subdir <- file.path(output_dir, paste0(prefix, match_mode))
  before_path   <- file.path(output_dir, "results_before_matching.csv")
  res_path      <- file.path(method_subdir, "results.csv")

  common_args <- list(
    group_col       = CONFIG$group_col,
    output_dir      = output_dir,
    treatment       = treatment,
    control         = control,
    match_mode      = match_mode,
    caliper_q       = CONFIG$caliper_q,
    caliper_ps_sd   = CONFIG$caliper_ps,
    all_covars      = cv$all,
    fact_covars     = cv$fact,
    demo_covars     = cv$demo,
    gower_demo_weight = CONFIG$gower_demo_w,
    gower_fact_weight = CONFIG$gower_fact_w,
    only_fact_vars  = only_fact_vars,
    bonferroni_k    = 30
  )

  cat("   [Pass 1] Main targets on full data\n")
  tryCatch(
    do.call(run_matching, c(list(df = df, targets_cols = TARGETS$main,
                                  binary_targets = BINARY_TARGETS), common_args)),
    error = function(e) cat("   Error pass 1:", conditionMessage(e), "\n")
  )

  pass1_results <- if (file.exists(res_path))    read.csv(res_path,    fileEncoding = "UTF-8") else NULL
  pass1_before  <- if (file.exists(before_path)) read.csv(before_path, fileEncoding = "UTF-8") else NULL

  if (!CONFIG$use_punishment_months) return(invisible(NULL))

  df_pm <- df %>% filter(punishment_months > 0)
  cat("   [Pass 2] punishment_months (N =", nrow(df_pm), ")\n")
  tryCatch(
    do.call(run_matching, c(list(df = df_pm, targets_cols = TARGETS$pm,
                                  binary_targets = NULL), common_args)),
    error = function(e) cat("   Error pass 2:", conditionMessage(e), "\n")
  )

  if (!is.null(pass1_results) && file.exists(res_path)) {
    write.csv(rbind(pass1_results, read.csv(res_path, fileEncoding = "UTF-8")),
              res_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
  if (!is.null(pass1_before) && file.exists(before_path)) {
    write.csv(rbind(pass1_before, read.csv(before_path, fileEncoding = "UTF-8")),
              before_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
}

# =============================================================================
# HELPER: process one district (one-vs-all)
# =============================================================================
run_one_district <- function(df, drug, target_district, cv, idx, total) {
  folder_name <- paste0("district_", gsub(" ", "_", target_district), "_vs_All_Others")
  out_dir     <- paste0("matched_outputs/", drug, "/", folder_name)

  cat("\n========================================\n")
  cat(sprintf("DISTRICT %d/%d: %s vs All Others\n", idx, total, target_district))
  cat("========================================\n")

  df_temp <- df %>%
    mutate(district_one_vs_all = if_else(district == target_district,
                                         target_district, CONFIG$control_label))

  n_t <- sum(df_temp$district_one_vs_all == target_district,       na.rm = TRUE)
  n_c <- sum(df_temp$district_one_vs_all == CONFIG$control_label,  na.rm = TRUE)
  cat(sprintf("Sample sizes: %s = %d, %s = %d\n",
              target_district, n_t, CONFIG$control_label, n_c))

  if (n_t < CONFIG$min_group_n || n_c < CONFIG$min_group_n) {
    cat("WARNING: Skipping due to small sample size\n")
    return(invisible(NULL))
  }

  for (mm in CONFIG$match_modes) {
    cat("\n--- match_mode:", mm, "---\n")
    tryCatch(
      run_matching_two_passes(
        df = df_temp, output_dir = out_dir,
        treatment = target_district, control = CONFIG$control_label,
        match_mode = mm, cv = cv, only_fact_vars = FALSE
      ),
      error = function(e) cat("Error in matching for", mm, ":", conditionMessage(e), "\n")
    )
  }
}

# =============================================================================
# HELPER: collect one-vs-all summary for a drug
# =============================================================================
collect_summary <- function(drug, districts) {
  cat("\n========================================\n")
  cat("COLLECTING ONE-VS-ALL SUMMARY RESULTS\n")
  cat("========================================\n")
  summary_results <- lapply(districts, function(target_district) {
    folder_name  <- paste0("district_", gsub(" ", "_", target_district), "_vs_All_Others")
    results_path <- paste0("matched_outputs/", drug, "/", folder_name, "/match_logit/results.csv")
    if (!file.exists(results_path)) return(NULL)
    tryCatch(
      read_csv(results_path, show_col_types = FALSE) %>%
        mutate(comparison = paste0(target_district, " vs All Others"),
               treatment  = target_district,
               control    = CONFIG$control_label),
      error = function(e) { cat("Could not read results for", folder_name, "\n"); NULL }
    )
  })
  summary_results <- Filter(Negate(is.null), summary_results)
  if (length(summary_results) == 0) return(invisible(NULL))
  all_results <- bind_rows(summary_results)
  summary_path <- paste0("matched_outputs/", drug, "/district_one_vs_all_summary.csv")
  write_csv(all_results, summary_path)
  cat("\nSummary saved to:", summary_path, "\n")

  # p_bonferroni already computed upstream (bonferroni_k = 30) in results.csv.
  # Significance stars use the corrected p-value.
  summary_table <- all_results %>%
    select(comparison, outcome, diff, std_error, p_value, p_bonferroni, d_cohens) %>%
    mutate(
      sig = case_when(
        p_bonferroni <= 0.001 ~ "***",
        p_bonferroni <= 0.01  ~ "**",
        p_bonferroni <= 0.05  ~ "*",
        p_bonferroni <= 0.1   ~ ".",
        TRUE                  ~ ""
      ),
      diff_fmt = sprintf("%.2f%s", diff, sig)
    )
  cat("\n=== SUMMARY: Treatment Effects by District (One vs All, Bonferroni-corrected) ===\n\n")
  print(summary_table, n = 50)
  invisible(all_results)
}

# =============================================================================
# MAIN LOOP
# =============================================================================
for (kind in CONFIG$kinds) {
  for (drug_name in CONFIG$drugs) {
    cv   <- drop_personal_covars(covars_for(kind, drug_name), kind)
    df   <- load_drug_data(kind, drug_name)
    drug <- paste0(kind, "/", drug_name)

    cat("\n=== Processing:", drug, "===\n")
    cat("\n=== District Distribution (", drug_name, ") ===\n")
    print(table(df$district))

    districts <- eligible_districts(df)
    cat("\nTotal One-vs-All comparisons:", length(districts), "\n")
    for (i in seq_along(districts))
      cat(sprintf("   %2d. %s vs All Others\n", i, districts[i]))

    for (i in seq_along(districts))
      run_one_district(df, drug, districts[i], cv, i, length(districts))

    collect_summary(drug, districts)

    cat("\n========================================\n")
    cat("All comparisons complete for", drug, "!\n")
    cat("========================================\n")
  }
}

# # --- Pair-based analysis ---
# cat("\n--- Analyzing pairs from CSV ---\n")
# tryCatch(
#   analyze_pairs_from_csv(
#     df           = df,
#     pairs_path   = paste0("data/pairs/", CONFIG$group_col, "/", drug_name, ".csv"),
#     output_dir   = file.path("matched_outputs", drug, "manual_pairs"),
#     group_col    = CONFIG$group_col,
#     id_col       = "original_case",
#     treatment    = CONFIG$treatment,
#     control      = CONFIG$control,
#     targets_cols = TARGETS$main,
#     all_covars   = cv$all
#   ),
#   error = function(e) cat("  Error in pair analysis:", conditionMessage(e), "\n")
# )


## one vs one

In [ ]:
# =============================================================================
# DISTRICT ANALYSIS SETUP
# =============================================================================

# =============================================================================
# ALL PAIRWISE DISTRICT COMPARISONS
# =============================================================================

run_matching_two_passes <- function(df, group_col, output_dir, treatment, control,
                                     match_mode, only_fact_vars = FALSE, ...) {
  prefix <- if (only_fact_vars) "fact_match_" else "match_"
  method_subdir <- file.path(output_dir, paste0(prefix, match_mode))
  before_path <- file.path(output_dir, "results_before_matching.csv")

  cat("  [Pass 1] Main targets on full data\n")
  tryCatch({
    run_matching(df = df, group_col = group_col, targets_cols = targets_main,
                 output_dir = output_dir, treatment = treatment, control = control,
                 match_mode = match_mode, only_fact_vars = only_fact_vars,
                 binary_targets = c("is_actual_sentence"), ...)
  }, error = function(e) cat("  Error pass 1:", conditionMessage(e), "\n"))

  pass1_results <- NULL
  res_path <- file.path(method_subdir, "results.csv")
  if (file.exists(res_path)) {
    pass1_results <- read.csv(res_path, fileEncoding = "UTF-8")
  }
  pass1_before <- NULL
  if (file.exists(before_path)) {
    pass1_before <- read.csv(before_path, fileEncoding = "UTF-8")
  }

  if (!use_punishment_months) return(invisible(NULL))

  df_pm <- df %>% filter(punishment_months > 0)
  cat("  [Pass 2] punishment_months on filtered data (N=", nrow(df_pm), ")\n")
  tryCatch({
    run_matching(df = df_pm, group_col = group_col, targets_cols = targets_pm,
                 output_dir = output_dir, treatment = treatment, control = control,
                 match_mode = match_mode, only_fact_vars = only_fact_vars,
                 binary_targets = NULL, ...)
  }, error = function(e) cat("  Error pass 2:", conditionMessage(e), "\n"))

  if (!is.null(pass1_results) && file.exists(res_path)) {
    pass2_res <- read.csv(res_path, fileEncoding = "UTF-8")
    merged <- rbind(pass1_results, pass2_res)
    write.csv(merged, res_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
  if (!is.null(pass1_before) && file.exists(before_path)) {
    pass2_before <- read.csv(before_path, fileEncoding = "UTF-8")
    merged_before <- rbind(pass1_before, pass2_before)
    write.csv(merged_before, before_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
}


targets_main <- c("min_punishment", "max_punishment", "is_actual_sentence")
targets_pm <- c("punishment_months")
use_punishment_months <- FALSE  # Set FALSE to skip punishment_months analysis

kind <- c("impute", "no_impute", "only_age") 
drugs <- c("cannabis","cocaine", "hashish")
kind <- c("impute") 
drugs <- c("cannabis")

for (kind_val in kind) {
  for (d in drugs) {
    # Define covariates - EXCLUDING district since it's the group variable
    all_covars_district <- c(
      "age", "marital_status", "year", "Court",
      "production_offence", "use_offence", "importation_offence",
      "laboratory", "self_use", "ethnic", "sold_to_agent", "total_amount",
      "regret", "confession", "Criminal_record", "judge_ethnic"
    )
    
    exact_vars <- c("total_amount")
    
    fact_covars <- c(
      "production_offence", "use_offence", "importation_offence",
      "laboratory", "self_use", "sold_to_agent", "total_amount","judge_ethnic"
    )
    
    demo_covars <- c(
      "age", "marital_status", "year", "ethnic",
      "regret", "confession", "Court", "Criminal_record"
    )
    
    # Adjust for kind = "only_age"
    if (kind_val == "only_age") {
      all_covars_district <- setdiff(all_covars_district, "marital_status")
      demo_covars <- setdiff(demo_covars, "marital_status")
    }
    
    # Adjust for non-cannabis drugs
    if (d != "cannabis") {
      all_covars_district <- setdiff(all_covars_district, "production_offence")
      fact_covars <- setdiff(fact_covars, "production_offence")
    }
    
    # Load data
    drug <- paste0(kind_val, "/", d)
    print(paste0("Processing drug: ", drug))
    
    df <- read_csv(
      paste0("data/", drug, ".csv"),
      locale = locale(encoding = "UTF-8"),
      show_col_types = FALSE
    )
    
    df <- df %>%
      filter(Criminal_record != "לא ידוע") %>%
      mutate(
        total_amount = total_amount / 1000,
        district = as.character(district) %>% str_squish(),
        age = as.numeric(age),
        is_actual_sentence = as.integer(is_actual_sentence)
      )
    df <- df %>%
      filter(!(ethnic %in% c("Anonymous", "Unknown")))
    cat("
=== District Distribution (", d, ") ===
")
    print(table(df$district))

    # Get districts with sufficient sample size
    min_n <- 40
    min_n <- 0

    districts <- df %>%
      filter(!is.na(district)) %>%
      count(district, name = "n") %>%
      filter(n >= min_n) %>%
      arrange(district) %>%
      pull(district)
    
    # Generate all pairs (combinations of 2)
    district_pairs <- tryCatch({
      if (length(districts) < 2) {
        cat("WARNING: Not enough districts (need at least 2, found", length(districts), ")\n")
        list()  # Return empty list
      } else {
        combn(districts, 2, simplify = FALSE)
      }
    }, error = function(e) {
      cat("ERROR creating district pairs:", e$message, "\n")
      list()  # Return empty list on error
    })    
    cat("\nTotal pairwise comparisons:", length(district_pairs), "\n\n")
    
    # Print all pairs
    cat("Pairs to compare:\n")
    for (i in seq_along(district_pairs)) {
      cat(sprintf("  %2d. %s vs %s\n", i, district_pairs[[i]][1], district_pairs[[i]][2]))
    }
    
    # =============================================================================
    # RUN MATCHING FOR ALL PAIRS
    # =============================================================================
    
    for (pair_idx in seq_along(district_pairs)) {
      
      dist1 <- district_pairs[[pair_idx]][1]
      dist2 <- district_pairs[[pair_idx]][2]
      
      # Create safe folder name (replace spaces with underscores)
      folder_name <- paste0("district_", gsub(" ", "_", dist1), "_vs_", gsub(" ", "_", dist2))
      
      cat("\n\n")
      cat("========================================\n")
      cat(sprintf("PAIR %d/%d: %s vs %s\n", pair_idx, length(district_pairs), dist1, dist2))
      cat("========================================\n")
      
      # Check sample sizes
      n1 <- sum(df$district == dist1, na.rm = TRUE)
      n2 <- sum(df$district == dist2, na.rm = TRUE)
      cat(sprintf("Sample sizes: %s = %d, %s = %d\n", dist1, n1, dist2, n2))
      
      # Skip if either group is too small
      if (n1 < 10 || n2 < 10) {
        cat("WARNING: Skipping due to small sample size (< 10)\n")
        next
      }
      
      # -----------------------------
      # Standard Matching (logit)
      # -----------------------------
      match_modes <- c("logit")
      
      for (mm in match_modes) {

        if (mm == "custom") {
          all_covars_district <- c(
              "age","year","Court","ethnic",
              'confession','Criminal_record','marital_status'
          )  
          }   

        cat("\n--- Running matching for match_mode:", mm, "---\n")
        tryCatch({
          run_matching_two_passes(
            df = df,
            group_col = "district",
            output_dir = paste0("matched_outputs/", drug, "/", folder_name),
            treatment = dist1,
            control = dist2,
            caliper_q = 0.9,
            caliper_ps_sd = 0.2,
            match_mode = mm,
            exact_vars = exact_vars,
            all_covars = all_covars_district,
            fact_covars = fact_covars,
            demo_covars = demo_covars,
            gower_demo_weight = 1,
            gower_fact_weight = 3,
            only_fact_vars = FALSE
          )
        }, error = function(e) {
          cat("Error in matching for", mm, ":", e$message, "\n")
        })
      }
      
      # -----------------------------
      # Matching with only fact variables
      # -----------------------------
      # cat("\n--- Running matching for only fact with match_mode: logit ---\n")
      # tryCatch({
      #   run_matching_two_passes(
      #     df = df,
      #     group_col = "district",
      #     output_dir = paste0("matched_outputs/", drug, "/", folder_name),
      #     treatment = dist1,
      #     control = dist2,
      #     caliper_q = 0.9,
      #     caliper_ps_sd = 0.2,
      #     match_mode = "logit",
      #     exact_vars = exact_vars,
      #     all_covars = all_covars_district,
      #     fact_covars = fact_covars,
      #     demo_covars = demo_covars,
      #     gower_demo_weight = 1,
      #     gower_fact_weight = 4,
      #     only_fact_vars = TRUE
      #   )
      # }, error = function(e) {
      #   cat("Error in matching for logit (fact only):", e$message, "\n")
      # })
      
      # -----------------------------
      # CEM(amount) + EXACT(others)
      # -----------------------------
      # cat("\n--- Running matching for CEM(amount) + EXACT(others) ---\n")
      # tryCatch({
      #   run_matching_two_passes(
      #     df = df,
      #     group_col = "district",
      #     output_dir = paste0("matched_outputs/", drug, "/", folder_name),
      #     treatment = dist1,
      #     control = dist2,
      #     caliper_q = 0.9,
      #     caliper_ps_sd = 0.2,
      #     match_mode = "exact_cem",
      #     exact_vars = exact_vars,
      #     all_covars = all_covars_district,
      #     fact_covars = fact_covars,
      #     demo_covars = demo_covars,
      #     gower_demo_weight = 1,
      #     gower_fact_weight = 4,
      #     only_fact_vars = TRUE
      #   )
      # }, error = function(e) {
      #   cat("Error in matching for EXACT/CEM:", e$message, "\n")
      # })
      
      # -----------------------------
      # no caliper
      # # -----------------------------
      # cat("\n--- Running matching for no caliper ---\n")
      # tryCatch({
      #   run_matching_two_passes(
      #     df = df,
      #     group_col = "district",
      #     output_dir = paste0("matched_outputs/", drug, "/", folder_name),
      #     treatment = dist1,
      #     control = dist2,
      #     caliper_q = 0.2,
      #     caliper_ps_sd = 0.2,
      #     match_mode = "no_caliper",
      #     exact_vars = exact_vars,
      #     all_covars = all_covars_district,
      #     fact_covars = fact_covars,
      #     demo_covars = demo_covars,
      #     gower_demo_weight = 1,
      #     gower_fact_weight = 3,
      #     only_fact_vars = FALSE
      #   )
      # }, error = function(e) {
      #   cat("Error in matching for no caliper:", e$message, "\n")
      # })
      
      # -----------------------------
      # Pair-based analysis (if pairs file exists)
      # -----------------------------
      # pairs_path <- paste0("data/pairs/district/", drug, "_", folder_name, ".csv")
      # if (file.exists(pairs_path)) {
      #   cat("\n--- Analyzing pairs from CSV ---\n")
      #   tryCatch({
      #     analyze_pairs_from_csv(
      #       df            = df,
      #       pairs_path    = pairs_path,
      #       output_dir    = paste0("matched_outputs/", drug, "/", folder_name, "/manual_pairs"),
      #       group_col     = "district",
      #       id_col        = "original_case",
      #       treatment     = dist1,
      #       control       = dist2,
      #       targets_cols  = c(targets_main, targets_pm),
      #       all_covars    = all_covars_district
      #     )
      #   }, error = function(e) {
      #     cat("Error in pair analysis:", e$message, "\n")
      #   })
      # }
    }
    
    # =============================================================================
    # SUMMARY TABLE - Collect all results
    # =============================================================================
    
    cat("\n\n")
    cat("========================================\n")
    cat("COLLECTING SUMMARY RESULTS\n")
    cat("========================================\n")
    
    summary_results <- list()
    
    for (pair_idx in seq_along(district_pairs)) {
      
      dist1 <- district_pairs[[pair_idx]][1]
      dist2 <- district_pairs[[pair_idx]][2]
      folder_name <- paste0("district_", gsub(" ", "_", dist1), "_vs_", gsub(" ", "_", dist2))
      
      # Try to read results from logit matching
      results_path <- paste0("matched_outputs/", drug, "/", folder_name, "/match_logit/results.csv")
      
      if (file.exists(results_path)) {
        tryCatch({
          res <- read_csv(results_path, show_col_types = FALSE)
          res <- res %>%
            mutate(
              comparison = paste0(dist1, " vs ", dist2),
              treatment = dist1,
              control = dist2
            )
          summary_results[[pair_idx]] <- res
        }, error = function(e) {
          cat("Could not read results for", folder_name, "\n")
        })
      }
    }
    
    # Combine all results
    if (length(summary_results) > 0) {
      all_results <- bind_rows(summary_results)
      
      # Save summary
      write_csv(all_results, paste0("matched_outputs/", drug, "/district_all_pairs_summary.csv"))
      
      cat("\nSummary saved to: matched_outputs/", drug, "/district_all_pairs_summary.csv\n")
      
      # Print summary table
      cat("\n=== SUMMARY: Treatment Effects by District Pair ===\n\n")
      
      summary_table <- all_results %>%
        select(comparison, outcome, diff, std_error, p_value, d_cohens) %>%
        mutate(
          sig = case_when(
            p_value <= 0.001 ~ "***",
            p_value <= 0.01 ~ "**",
            p_value <= 0.05 ~ "*",
            p_value <= 0.1 ~ ".",
            TRUE ~ ""
          ),
          diff_fmt = sprintf("%.2f%s", diff, sig)
        )
      
      print(summary_table, n = 50)
    }
    
    cat("\n\n========================================\n")
    cat("All pairwise district comparisons complete for", drug, "!\n")
    cat("========================================\n")
  }
}


# year

In [ ]:
# =============================================================================
# TIME-PERIOD ANALYSIS (Corona / War) — general + by-ethnic-group
# =============================================================================

CONFIG <- list(
  kinds       = c("no_impute","no_personal"),
  drugs       = c("cannabis", "cocaine", "hashish"),
  match_modes = c("logit"),
  min_n       = 10,
  caliper_q   = 0.9,
  caliper_ps  = 0.2,
  gower_demo_w = 1,
  gower_fact_w = 3,
  war_start   = as.Date("2023-10-07"),
  corona_start = as.Date("2020-03-01")
)

TARGETS        <- c("min_punishment", "max_punishment", "is_actual_sentence")
BINARY_TARGETS <- c("is_actual_sentence")

PERIOD_LABELS <- list(
  corona_period = list(control = "pre_corona", treatment = "post_corona"),
  war_period    = list(control = "pre_war",    treatment = "post_war")
)

# =============================================================================
# HELPER: covariate sets
# =============================================================================
covars_time <- function() {
  list(
    all = c("district","marital_status","Court","age",
            "production_offence","use_offence","importation_offence",
            "laboratory","self_use","ethnic","sold_to_agent","total_amount",
            "regret","confession","Criminal_record","judge_ethnic"),
    fact = c("production_offence","use_offence","importation_offence",
             "laboratory","self_use","sold_to_agent","total_amount"),
    demo = c("district","age","marital_status","ethnic",
             "regret","confession","Court","Criminal_record","judge_ethnic")
  )
}

# =============================================================================
# HELPER: load + clean one drug's data
# =============================================================================
load_drug_df <- function(kind, drug) {
  read_csv(paste0("data/", data_dir_for(kind), "/", drug, ".csv"),
           locale = locale(encoding = "UTF-8"), show_col_types = FALSE) %>%
    filter(Criminal_record != "לא ידוע") %>%
    mutate(
      total_amount       = total_amount / 1000,
      district           = str_squish(as.character(district)),
      age                = as.numeric(age),
      year               = as.numeric(as.character(year)),
      is_actual_sentence = as.integer(is_actual_sentence),
      Date               = dmy(Date),
      war_period         = if_else(Date >= CONFIG$war_start, 1L, 0L),
      corona_period      = if_else(Date >= CONFIG$corona_start, 1L, 0L)
    ) %>%
    filter(!(ethnic %in% c("Anonymous", "Unknown")))
}

# =============================================================================
# HELPER: one time-period comparison
# =============================================================================
run_time_comparison <- function(df, period_col, comparison_name, drug, kind,
                                 binary_targets = BINARY_TARGETS) {

  if (!period_col %in% names(df)) {
    cat("WARNING: column '", period_col, "' not found. Skipping.\n")
    return(invisible(NULL))
  }
  if (!period_col %in% names(PERIOD_LABELS)) {
    stop("Unknown period_col: ", period_col)
  }

  lbl       <- PERIOD_LABELS[[period_col]]
  group_col <- paste0(period_col, "_group")

  df_local <- df %>%
    mutate(!!group_col := case_when(
      .data[[period_col]] == 1 ~ lbl$treatment,
      .data[[period_col]] == 0 ~ lbl$control,
      TRUE                     ~ NA_character_
    ))

  cat("\n========================================\n")
  cat("TIME COMPARISON:", comparison_name, "| period:", period_col, "\n")
  cat("========================================\n")
  print(table(df_local[[group_col]], useNA = "ifany"))

  n_treat <- sum(df_local[[group_col]] == lbl$treatment, na.rm = TRUE)
  n_ctrl  <- sum(df_local[[group_col]] == lbl$control,   na.rm = TRUE)
  cat(sprintf("\nSample sizes: %s = %d, %s = %d\n",
              lbl$treatment, n_treat, lbl$control, n_ctrl))

  if (n_treat < CONFIG$min_n || n_ctrl < CONFIG$min_n) {
    cat("\nWARNING: Skipping — insufficient sample (min:", CONFIG$min_n, ")\n")
    return(invisible(NULL))
  }

  cv      <- drop_personal_covars(covars_time(), kind)
  out_dir <- file.path("matched_outputs", drug, comparison_name)

  # When the period IS the treatment, war_period must not enter as a covariate
  # or as an interaction term (it would be collinear with `treated`).
  interaction_var <- if (period_col == "war_period") "war_period" else "ethnic"

  for (mm in CONFIG$match_modes) {
    cat("\n--- match_mode:", mm, "---\n")
    tryCatch(
      run_matching(
        df             = df_local,
        group_col      = group_col,
        targets_cols   = TARGETS,
        output_dir     = out_dir,
        treatment      = lbl$treatment,
        control        = lbl$control,
        caliper_q      = CONFIG$caliper_q,
        caliper_ps_sd  = CONFIG$caliper_ps,
        match_mode     = mm,
        all_covars     = cv$all,
        fact_covars    = cv$fact,
        demo_covars    = cv$demo,
        gower_demo_weight = CONFIG$gower_demo_w,
        gower_fact_weight = CONFIG$gower_fact_w,
        only_fact_vars = FALSE,
        binary_targets = binary_targets,
        interaction_var = interaction_var
      ),
      error = function(e) cat("Error in matching for", mm, ":", conditionMessage(e), "\n")
    )
  }
  invisible(TRUE)
}

# =============================================================================
# MAIN — Part 1: general time comparisons (corona + war), pooled
# =============================================================================
for (kind in CONFIG$kinds) {
  for (drug_name in CONFIG$drugs) {
    drug <- paste0(kind, "/", drug_name)
    cat("\n=== Processing:", drug, "===\n")
    df <- load_drug_df(kind, drug_name)

    for (time_col in c("corona_period", "war_period")) {
      comparison_name <- gsub("_period$", "", time_col)
      cat("\n--- Comparison:", comparison_name, "for", drug, "---\n")
      tryCatch(
        run_time_comparison(df, time_col, comparison_name, drug, kind),
        error = function(e) cat("ERROR in", time_col, ":", conditionMessage(e), "\n")
      )
    }
    cat("\n=== Time comparisons complete for", drug, "===\n")
  }
}

# =============================================================================
# MAIN — Part 2: war comparison, split by ethnic group (Arab / Jewish)
# =============================================================================
for (kind in CONFIG$kinds) {
  for (drug_name in CONFIG$drugs) {
    drug <- paste0(kind, "/", drug_name)
    cat("\n=== Processing (by ethnic):", drug, "===\n")
    df <- load_drug_df(kind, drug_name) %>% filter(ethnic %in% c("Arab", "Jewish"))

    for (ethnic_group in c("Arab", "Jewish")) {
      df_eth <- df %>% filter(ethnic == ethnic_group)
      n_pre  <- sum(df_eth$war_period == 0, na.rm = TRUE)
      n_war  <- sum(df_eth$war_period == 1, na.rm = TRUE)
      cat("\n====", drug, "|", ethnic_group, "| pre:", n_pre, "war:", n_war, "====\n")

      if (n_pre < CONFIG$min_n || n_war < CONFIG$min_n) {
        cat("  Skipping — insufficient sample size\n")
        next
      }
      tryCatch(
        run_time_comparison(df_eth, "war_period",
                            paste0("war/", ethnic_group), drug, kind),
        error = function(e) cat("  ERROR:", conditionMessage(e), "\n")
      )
    }
  }
}

In [ ]:
# =============================================================================
# run_ethnic_by_war_period_stratified_drugs.R
# PSM stratified by sector (Arab / Jewish): pre_war vs. war, per drug.
# Q: Did sentencing change after the war — separately within each sector?
# =============================================================================

CONFIG <- list(
  kinds       = c("no_impute","no_personal"),
  drugs       = c("cannabis", "cocaine", "hashish"),
  sectors     = c("Arab", "Jewish"),
  war_cutoff  = as.Date("2023-10-07"),
  min_n       = 10,
  caliper_ps  = 0.2,
  match_mode  = "logit"
)

TARGETS        <- c("min_punishment", "max_punishment", "is_actual_sentence")
BINARY_TARGETS <- c("is_actual_sentence")

# =============================================================================
# HELPER: covariate sets (ethnic excluded — sector is fixed within each loop)
# =============================================================================
covars_sets <- function() {
  list(
    all = c("district","age","Court",
            "production_offence","use_offence","importation_offence",
            "laboratory","self_use","sold_to_agent","total_amount",
            "actual_damage","premeditation",
            "regret","confession","Criminal_record","marital_status","judge_ethnic"),
    fact = c("production_offence","use_offence","importation_offence",
             "laboratory","self_use","sold_to_agent","total_amount",
             "actual_damage","premeditation"),
    demo = c("district","age","Court",
             "regret","confession","Criminal_record","marital_status","judge_ethnic")
  )
}

# =============================================================================
# HELPER: load + clean one drug's data
# =============================================================================
load_drug_df <- function(kind, drug) {
  read_csv(paste0("data/", data_dir_for(kind), "/", drug, ".csv"),
           locale = locale(encoding = "UTF-8"), show_col_types = FALSE) %>%
    filter(Criminal_record != "לא ידוע") %>%
    mutate(
      district           = str_squish(as.character(district)),
      age                = as.numeric(age),
      year               = as.numeric(as.character(year)),
      is_actual_sentence = as.integer(is_actual_sentence),
      Date               = dmy(Date),
      war_period         = if_else(Date >= CONFIG$war_cutoff, "war", "pre_war"),
      total_amount       = total_amount / 1000
    ) %>%
    filter(ethnic %in% c("Arab", "Jewish")) %>%
    filter(!(judge_ethnic %in% c("Anonymous", "Unknown")))
}

# =============================================================================
# HELPER: drop covariates that are constant within a subgroup
# =============================================================================
keep_varying_covars <- function(covars, df_sub) {
  covars[sapply(covars, function(v) {
    if (!v %in% names(df_sub)) return(FALSE)
    length(unique(na.omit(as.character(df_sub[[v]])))) >= 2
  })]
}

# =============================================================================
# HELPER: run one sector × drug matching
# =============================================================================
run_sector <- function(df_sector, drug, sector, cv, kind) {
  cat("\n  Sector:", sector, "\n")
  print(df_sector %>% count(war_period))

  n_pre <- sum(df_sector$war_period == "pre_war", na.rm = TRUE)
  n_war <- sum(df_sector$war_period == "war",     na.rm = TRUE)
  if (n_pre < CONFIG$min_n || n_war < CONFIG$min_n) {
    cat("  Skipping — pre_war:", n_pre, "war:", n_war, "\n")
    return(NULL)
  }

  covars_sector <- keep_varying_covars(cv$all, df_sector)
  out_dir <- paste0("matched_outputs/", kind, "/", drug,
                    "/time_by_sector/", sector)

  tryCatch(
    run_matching(
      df              = df_sector,
      group_col       = "war_period",
      targets_cols    = TARGETS,
      output_dir      = out_dir,
      treatment       = "war",
      control         = "pre_war",
      caliper_ps_sd   = CONFIG$caliper_ps,
      match_mode      = CONFIG$match_mode,
      all_covars      = covars_sector,
      fact_covars     = cv$fact,
      demo_covars     = cv$demo,
      binary_targets  = BINARY_TARGETS,
      interaction_var = "war_period"
      # war_interaction left FALSE: war_period IS the treatment here
    ),
    error = function(e) cat("  ERROR:", conditionMessage(e), "\n")
  )

  res_path <- file.path(out_dir, "match_logit", "results.csv")
  if (!file.exists(res_path)) return(NULL)
  read_csv(res_path, show_col_types = FALSE) %>%
    mutate(kind = kind, drug = drug, sector = sector)
}

# =============================================================================
# MAIN LOOP
# =============================================================================
all_results <- list()

for (kind in CONFIG$kinds) {
  for (drug in CONFIG$drugs) {
    df <- load_drug_df(kind, drug)
    cv <- drop_personal_covars(covars_sets(), kind)

    cat("\n====", kind, "/", drug, "====\n")
    print(df %>% count(ethnic, war_period) %>% arrange(ethnic, war_period))

    for (sector in CONFIG$sectors) {
      df_sector <- df %>% filter(ethnic == sector)
      res <- run_sector(df_sector, drug, sector, cv, kind)
      if (!is.null(res)) all_results[[paste(kind, drug, sector, sep = "_")]] <- res
    }
  }
}

# =============================================================================
# SUMMARY TABLE
# =============================================================================
if (length(all_results) == 0) {
  stop("No results collected — check that run_matching succeeded for at least one combination.")
}

summary_table <- bind_rows(all_results) %>%
  select(any_of(c("kind","drug","sector","outcome","is_binary",
                  "mean_treatment","mean_control",
                  "diff","ci_low","ci_high","p_value",
                  "d_cohens","n_treatment_unique","n_control_unique"))) %>%
  arrange(kind, drug, outcome, sector)

print(summary_table)

# Robustness – alternative matching for significant district findings (SM, Section 3.3.1)

In [ ]:
# =============================================================================
# ROBUSTNESS — ALTERNATIVE MATCHING FOR SIGNIFICANT DISTRICT FINDINGS
# Same data and covariates as the primary one-vs-all district analysis;
# only the matching algorithm changes (Gower distance, exact on offense vars).
# =============================================================================
ROBUST <- list(
  kind          = "impute",
  districts     = list(cannabis = c("Center", "Jerusalem")),
  match_specs   = list(list(mode = "gower",     only_fact = FALSE),
                       list(mode = "exact_cem", only_fact = TRUE)),
  control_label = "All_Others",
  caliper_q     = 0.9,
  caliper_ps    = 0.2,
  gower_demo_w  = 1,
  gower_fact_w  = 3,
  bonferroni_k  = 30
)

# Covariates as in the primary district analysis (district excluded — it's the group)
robust_district_covars <- function(drug) {
  cv <- list(
    all = c("age","marital_status","year","Court",
            "production_offence","use_offence","importation_offence",
            "laboratory","self_use","ethnic","sold_to_agent","total_amount",
            "regret","confession","Criminal_record","judge_ethnic"),
    fact = c("production_offence","use_offence","importation_offence",
             "laboratory","self_use","sold_to_agent","total_amount","judge_ethnic"),
    demo = c("age","marital_status","year","ethnic",
             "regret","confession","Court","Criminal_record")
  )
  if (drug != "cannabis") {
    cv$all  <- setdiff(cv$all,  "production_offence")
    cv$fact <- setdiff(cv$fact, "production_offence")
  }
  cv
}

robust_load_drug <- function(kind, drug) {
  read_csv(paste0("data/", kind, "/", drug, ".csv"),
           locale = locale(encoding = "UTF-8"), show_col_types = FALSE) %>%
    filter(Criminal_record != "לא ידוע") %>%
    mutate(
      total_amount       = total_amount / 1000,
      district           = str_squish(as.character(district)),
      age                = as.numeric(age),
      is_actual_sentence = as.integer(is_actual_sentence)
    ) %>%
    filter(!(ethnic %in% c("Anonymous", "Unknown"))) %>%
    filter(!is.na(district) & district != "")
}

for (drug in names(ROBUST$districts)) {
  df <- robust_load_drug(ROBUST$kind, drug)
  cv <- robust_district_covars(drug)

  for (target_district in ROBUST$districts[[drug]]) {
    df_temp <- df %>%
      mutate(district_one_vs_all = if_else(district == target_district,
                                           target_district, ROBUST$control_label))
    out_dir <- paste0("matched_outputs/", ROBUST$kind, "/", drug, "/district_",
                      gsub(" ", "_", target_district), "_vs_All_Others")

    for (spec in ROBUST$match_specs) {
      cat("\n===", drug, "|", target_district, "vs All Others |", spec$mode,
          if (spec$only_fact) "(offense vars only)" else "", "===\n")
      tryCatch(
        run_matching(
          df = df_temp, group_col = "district_one_vs_all",
          targets_cols = c("min_punishment", "max_punishment", "is_actual_sentence"),
          output_dir = out_dir,
          treatment = target_district, control = ROBUST$control_label,
          caliper_q = ROBUST$caliper_q, caliper_ps_sd = ROBUST$caliper_ps,
          match_mode = spec$mode,
          all_covars = cv$all, fact_covars = cv$fact, demo_covars = cv$demo,
          gower_demo_weight = ROBUST$gower_demo_w, gower_fact_weight = ROBUST$gower_fact_w,
          only_fact_vars = spec$only_fact,
          binary_targets = "is_actual_sentence",
          bonferroni_k = ROBUST$bonferroni_k
        ),
        error = function(e) cat("  ERROR:", conditionMessage(e), "\n")
      )
    }
  }
}